In [3]:
import os
print(os.getcwd())

C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project


In [2]:
os.chdir(r'C:\Users\otmil\OneDrive - United Nations\Desktop\Career\Great Ape Conservation Project')

In [5]:
import geopandas as gpd
import pandas as pd

part0 = gpd.read_file('data/WDPA_WDOECM_Aug2026_Public_GNQ_shp-polygons.shp')
part1 = gpd.read_file('data/WDPA_WDOECM_Aug2026_Public_GNQ_shp-polygons 1.shp')
part2 = gpd.read_file('data/WDPA_WDOECM_Aug2026_Public_GNQ_shp-polygons 2.shp')


In [6]:

print(part0.shape, part1.shape, part2.shape)
print(part0.crs)

(6, 34) (6, 34) (4, 34)
EPSG:4326


In [7]:
eg_wdpa = pd.concat([part0, part1, part2], ignore_index=True)
eg_wdpa = gpd.GeoDataFrame(eg_wdpa, crs=part0.crs)

print(eg_wdpa.shape)
print(eg_wdpa.columns.tolist())

(16, 34)
['SITE_ID', 'SITE_PID', 'SITE_TYPE', 'NAME_ENG', 'NAME', 'DESIG', 'DESIG_ENG', 'DESIG_TYPE', 'IUCN_CAT', 'INT_CRIT', 'REALM', 'REP_M_AREA', 'GIS_M_AREA', 'REP_AREA', 'GIS_AREA', 'NO_TAKE', 'NO_TK_AREA', 'STATUS', 'STATUS_YR', 'GOV_TYPE', 'GOVSUBTYPE', 'OWN_TYPE', 'OWNSUBTYPE', 'MANG_AUTH', 'MANG_PLAN', 'VERIF', 'METADATAID', 'PRNT_ISO3', 'ISO3', 'SUPP_INFO', 'CONS_OBJ', 'INLND_WTRS', 'OECM_ASMT', 'geometry']


In [10]:
ape_ranges = gpd.read_file('data/Ape_ranges.shp')
countries = gpd.read_file('data/ne_10m_admin_0_countries.shp')
countries = countries.to_crs(ape_ranges.crs)

ape_ranges_by_country = gpd.overlay(ape_ranges, countries[['NAME_LONG', 'geometry']], how='intersection')
ape_ranges_by_country = ape_ranges_by_country.rename(columns={'NAME_LONG': 'country'})

print(ape_ranges_by_country['country'].unique())

['Nigeria' 'Cameroon' 'Republic of the Congo'
 'Democratic Republic of the Congo' 'Central African Republic' 'Angola'
 'Gabon' 'Equatorial Guinea' 'South Sudan' 'Tanzania' 'Burundi' 'Rwanda'
 'Uganda' 'Sierra Leone' 'Guinea' 'Liberia' "Côte d'Ivoire" 'Mali'
 'Senegal' 'Ghana' 'Guinea-Bissau' 'Malaysia' 'Indonesia']


In [8]:
print(eg_wdpa[['NAME_ENG', 'DESIG_ENG', 'STATUS']])

                                 NAME_ENG  \
0                       Estuario del Muni   
1                          Pico de Basilé   
2                              Monte Alén   
3                          Altos de Nsork   
4                                 Annobón   
5                         Caldera de Luba   
6                            Playa Nendyi   
7                             Piedra Bere   
8                             Piedra Nzas   
9                               Rio Campo   
10                           Punta Llende   
11                     Corisco y Elobeyes   
12                          Monte Temelón   
13                        Isla de Annobón   
14  Reserva Natural del Estuario del Muni   
15                       Río Ntem o Campo   

                                            DESIG_ENG      STATUS  
0                                      Nature reserve  Designated  
1                                       National Park  Designated  
2                             

In [11]:
eg_range = ape_ranges_by_country[ape_ranges_by_country['country'] == 'Equatorial Guinea'].dissolve()

eg_wdpa_filtered = eg_wdpa[eg_wdpa['STATUS'] != 'Proposed']
eg_pas_dissolved = eg_wdpa_filtered.dissolve()

eg_overlay = gpd.overlay(eg_range[['geometry']], eg_pas_dissolved[['geometry']], how='intersection')
eg_overlay_ea = eg_overlay.to_crs('EPSG:6933')
eg_pa_km2 = eg_overlay_ea.geometry.area.sum() / 1e6

eg_range_ea = eg_range.to_crs('EPSG:6933')
eg_range_km2 = eg_range_ea.geometry.area.sum() / 1e6

eg_pct = (eg_pa_km2 / eg_range_km2) * 100
print(f"Equatorial Guinea: {eg_range_km2:.1f} km² range, {eg_pa_km2:.1f} km² protected, {eg_pct:.1f}% covered")

Equatorial Guinea: 24609.4 km² range, 4173.4 km² protected, 17.0% covered


In [13]:
coverage_df = pd.read_csv('data/pa_coverage_by_country.csv')
print(coverage_df)

                             country  range_area_ha    pa_area_ha  pct_covered
0                           Cameroon   2.349081e+07  2.845281e+06    12.112316
1           Central African Republic   1.277954e+07  1.840348e+06    14.400736
2   Democratic Republic of the Congo   1.230251e+08  2.108845e+07    17.141581
3                              Gabon   2.547419e+07  5.297196e+06    20.794366
4                            Nigeria   3.055497e+06  1.713522e+06    56.079971
5              Republic of the Congo   2.467262e+07  1.006223e+07    40.782991
6                             Rwanda   2.551466e+05  1.230639e+05    48.232638
7                            Burundi   6.179705e+05  5.092588e+04     8.240827
8                           Tanzania   1.660321e+06  4.003476e+05    24.112658
9                             Uganda   2.025613e+06  8.227782e+05    40.618726
10                 Equatorial Guinea   2.460940e+06  1.665481e+03     0.067677
11                         Indonesia   2.355644e+07 

In [14]:
eg_row = pd.DataFrame([{
    'country': 'Equatorial Guinea',
    'range_area_ha': eg_range_km2 * 100,
    'pa_area_ha': eg_pa_km2 * 100,
    'pct_covered': eg_pct
}])

coverage_df = pd.concat([coverage_df, eg_row], ignore_index=True)
print(coverage_df)

                             country  range_area_ha    pa_area_ha  pct_covered
0                           Cameroon   2.349081e+07  2.845281e+06    12.112316
1           Central African Republic   1.277954e+07  1.840348e+06    14.400736
2   Democratic Republic of the Congo   1.230251e+08  2.108845e+07    17.141581
3                              Gabon   2.547419e+07  5.297196e+06    20.794366
4                            Nigeria   3.055497e+06  1.713522e+06    56.079971
5              Republic of the Congo   2.467262e+07  1.006223e+07    40.782991
6                             Rwanda   2.551466e+05  1.230639e+05    48.232638
7                            Burundi   6.179705e+05  5.092588e+04     8.240827
8                           Tanzania   1.660321e+06  4.003476e+05    24.112658
9                             Uganda   2.025613e+06  8.227782e+05    40.618726
10                 Equatorial Guinea   2.460940e+06  1.665481e+03     0.067677
11                         Indonesia   2.355644e+07 

In [15]:
coverage_df.to_csv('data/pa_coverage_by_country.csv', index=False)

In [17]:
total_range = coverage_df['range_area_ha'].sum()
total_pa = coverage_df['pa_area_ha'].sum()
overall_pct = (total_pa / total_range) * 100
print(total_range)
print(total_pa)
print(f"Overall: {overall_pct:.1f}%")

250450183.26968655
49162569.16503533
Overall: 19.6%


In [18]:
coverage_df = coverage_df[~((coverage_df['country'] == 'Equatorial Guinea') & (coverage_df['pct_covered'] < 1))]
print(coverage_df)

                             country  range_area_ha    pa_area_ha  pct_covered
0                           Cameroon   2.349081e+07  2.845281e+06    12.112316
1           Central African Republic   1.277954e+07  1.840348e+06    14.400736
2   Democratic Republic of the Congo   1.230251e+08  2.108845e+07    17.141581
3                              Gabon   2.547419e+07  5.297196e+06    20.794366
4                            Nigeria   3.055497e+06  1.713522e+06    56.079971
5              Republic of the Congo   2.467262e+07  1.006223e+07    40.782991
6                             Rwanda   2.551466e+05  1.230639e+05    48.232638
7                            Burundi   6.179705e+05  5.092588e+04     8.240827
8                           Tanzania   1.660321e+06  4.003476e+05    24.112658
9                             Uganda   2.025613e+06  8.227782e+05    40.618726
11                         Indonesia   2.355644e+07  3.569432e+06    15.152682
12                          Malaysia   4.915046e+06 

In [19]:
coverage_df.to_csv('data/pa_coverage_by_country.csv', index=False)

In [21]:
total_range = coverage_df['range_area_ha'].sum()
total_pa = coverage_df['pa_area_ha'].sum()
overall_pct = (total_pa / total_range) * 100
print(f"Overall: {overall_pct:.1f}%")
print(total_range)
print(total_pa)

Overall: 19.8%
247989242.92533898
49160903.68386776
